
# 🧱 Databricks Course — Day 2 Notes
> **Focus:** Cluster Configuration, Notebooks, Magic Commands, DBUtils, Git Integration & Debugging

---

## 1. 🖥️ Cluster Types — Single Node vs Multi Node

| | Single Node | Multi Node |
|---|---|---|
| **Driver** | Yes (only node) | 1 dedicated Driver node |
| **Workers** | ❌ None | 1 or more Worker nodes |
| **Scalability** | ❌ Not scalable | ✅ Horizontal scaling |
| **Use Case** | Dev, small data, testing | Production, large-scale workloads |

### How Multi-Node Works:
```
Driver Node  ──→  distributes tasks  ──→  Worker Node 1
                                     ──→  Worker Node 2
                                     ──→  Worker Node N
```
- **Driver** — coordinates the job, collects results, runs your notebook code
- **Workers** — do the actual heavy Spark computation in parallel

**⚠️ Production Tip:** Never use Single Node for production pipelines. If a single node fails, everything fails. Multi-node gives you fault tolerance — if one worker dies, Spark reschedules the task on another.

---

## 2. 🔐 Access Modes — 3 Types

| Access Mode | Users | Available On | Supported Languages | Best For |
|---|---|---|---|---|
| **Single User (Dedicated)** | Only 1 user | Standard & Premium | Python, SQL, Scala, R | Personal dev/test clusters |
| **Shared** | Multiple users | Premium only | Python, SQL | Production — team clusters |
| **No Isolation Shared** | Multiple users | Standard & Premium | Python, SQL, Scala, R | Legacy / specific use cases |

### 🏆 Production Recommendation → **Shared Access Mode**
- Multiple team members on one cluster = **cost savings** (no idle clusters per person)
- Isolation between users — one user's bad query won't crash another's session
- Requires **Premium** plan — worth it for any serious production environment

**⚠️ Production Tip:** No Isolation Shared is risky in production — one runaway query can kill the entire cluster for everyone. Avoid it unless you have a very specific reason.

---

## 3. ⚙️ Databricks Runtime (DBR)

| Runtime | Includes | Use Case |
|---|---|---|
| **Databricks Runtime (DBR)** | Apache Spark + optimized libraries + Photon | General ETL, SQL, Streaming |
| **Databricks Runtime ML** | Everything in DBR + ML libraries (TensorFlow, PyTorch, scikit-learn, XGBoost) | Machine Learning workloads |

**⚠️ Production Tip:** Always pin a specific **LTS** runtime version (e.g., `13.3 LTS`) in production. LTS = Long Term Support — stable, patched, no surprise breaking changes. Avoid non-LTS runtimes in production pipelines.

---

## 4. ⏱️ Auto Termination

- Cluster **automatically shuts down** after X minutes of inactivity
- **Default:** 120 minutes for Single Node & Standard clusters
- **Configurable range:** 10 min → 43,200 min (30 days)

**⚠️ Production Tip:** Set aggressive auto-termination on dev/test clusters (30–60 min) to avoid burning cloud budget overnight. For production **job clusters**, auto-termination isn't needed — they spin up for a job and terminate automatically when done.

---

## 5. 📈 Auto Scaling

- You set **min and max** worker nodes
- Databricks **automatically scales** between min and max based on current workload
- Scales up when load is high → scales down when idle → **saves cost**

### Spot Instances:
- **What they are:** Unused/spare VMs in the cloud — up to 70–90% cheaper than on-demand
- **Risk:** Cloud provider can reclaim them at any time
- **Databricks fallback:** If spot instances are reclaimed, Databricks tries to acquire new ones or switches to **on-demand instances** automatically

**⚠️ Production Tip:** Use spot instances for **worker nodes only**, never for the driver. If the driver dies mid-job, the entire job fails. Workers are replaceable; the driver is not.
> ✅ Best pattern: **On-demand driver + Spot workers**

---

## 6. 🖧 Cluster VM Type / Size — When to Use What

| Instance Type | Optimized For | Best Use Case |
|---|---|---|
| **Memory Optimized** | High RAM | ML training, large joins, caching heavy datasets |
| **Compute Optimized** | High CPU | Streaming apps, distributed analytics, data science |
| **Storage Optimized** | High disk I/O & throughput | Large reads/writes, shuffle-heavy jobs, data ingestion |
| **General Purpose** | Balanced CPU + RAM | Standard ETL, enterprise analytics, dashboards |
| **GPU Accelerated** | GPU compute | Deep learning (TensorFlow, PyTorch) — data + compute intensive |

**⚠️ Production Tip:** Start with General Purpose, then tune based on your bottleneck. If jobs are slow due to joins/shuffles → Memory Optimized. Streaming pipeline → Compute Optimized. Profile first, don't over-provision blindly.

---

## 7. 📋 Cluster Policy

- Admins define **templates & guardrails** for cluster creation
- Controls what users can set: instance types, max DBUs, auto-termination, allowed runtimes
- Users can only create clusters within the allowed policy bounds

**⚠️ Production Tip:** Always enforce cluster policies in org environments. Without them, developers spin up giant expensive clusters "just to test" and forget to terminate them. Policies protect the cloud bill.

---

## 8. 📓 Databricks Notebooks — Basics

- Browser-based interactive coding environment
- Supports multiple languages in the **same notebook** via magic commands
- GitHub connectivity available for version control (covered in section 11)

---

## 9. ✨ Magic Commands

Magic commands switch behavior for a **single cell only** — they don't affect the rest of the notebook.

| Command | What it does | Example / Note |
|---|---|---|
| `%python` | Run cell as Python | Default in Python notebooks |
| `%sql` | Run cell as SQL | `%sql SELECT * FROM my_table` |
| `%scala` | Run cell as Scala | — |
| `%r` | Run cell as R | — |
| `%md` | Render cell as Markdown | For documentation inside notebooks |
| `%sh` | Run shell commands | Runs on **driver node only** — `%sh ps` |
| `%pip` | Install Python libraries | `%pip install faker` / `%pip list` |
| `%run` | Execute another notebook | Imports its variables & functions into current notebook |
| `%fs` | File system commands | `%fs ls /mnt/data/` |

### `%run` — Practical Use Case:
```python
# notebook_config.py has environment variables and shared functions
# In your pipeline notebook:
%run ./notebook_config
# Now all variables and functions from config notebook are available here
```

**⚠️ Production Tip:** Use `%run` to create a **config notebook** with env variables (dev/staging/prod paths, secrets scope names, table names) and run it at the top of every pipeline notebook. Clean, DRY, and easy to maintain.

---

## 10. 🛠️ DBUtils — Databricks Utilities

> `dbutils.help()` — shows all available utilities and their methods

**Magic commands** = cell-level language switches (interactive)
**DBUtils** = Python objects for programmatic control of the environment (used in pipelines)

| Utility | Commands | What it does | When to use |
|---|---|---|---|
| **File System** | `dbutils.fs.*` | List, copy, move, delete files in cloud storage | Moving files between Bronze/Silver/Gold layers |
| **Secrets** | `dbutils.secrets.*` | Securely fetch credentials stored in secret scopes | Always — never hardcode secrets in notebooks |
| **Widgets** | `dbutils.widgets.*` | Create input parameters for notebooks | Parameterizing notebooks for different runs/envs |
| **Notebook Workflow** | `dbutils.notebook.*` | Run notebooks from other notebooks, pass results back | Orchestrating multi-notebook pipelines |

### Quick Examples:

```python
# File System
dbutils.fs.ls("/mnt/datalake/bronze/")
dbutils.fs.cp("/mnt/source/file.csv", "/mnt/dest/file.csv")

# Secrets — NEVER hardcode passwords
password = dbutils.secrets.get(scope="my-scope", key="db-password")

# Widgets — parameterized notebooks
dbutils.widgets.text("run_date", "2024-01-01")
run_date = dbutils.widgets.get("run_date")

# Notebook Workflow — orchestrate child notebooks
result = dbutils.notebook.run("./child_notebook", timeout_seconds=300, arguments={"env": "prod"})
```

### Magic Commands vs DBUtils — When to use which?

| | Magic Commands | DBUtils |
|---|---|---|
| **Nature** | Interactive, cell-level | Programmatic Python objects |
| **Best for** | Ad-hoc queries, shell commands, markdown | Automation, pipelines, secrets, parameters |
| **In production pipelines** | Rarely | Always |

**⚠️ Production Tip:** Parameterize every production notebook using `dbutils.widgets`. This lets your orchestration tool (Databricks Workflows, ADF, Airflow) pass dynamic values like `run_date`, `environment`, `table_name` at runtime — no hardcoding anything.

---

## 11. 🗂️ Databricks Git Folders (Repos)

### Problem with Built-in Notebook Version History:
- Tracks only a **single notebook** — not the whole project
- **No branching** — can't have dev / staging / main branches
- **No CI/CD** integration
- Very basic — not real version control

### With Git Integration you get:
- ✅ **Holistic version control** — track all notebooks, configs, and scripts together
- ✅ **Team collaboration** — PRs, code reviews, feature branches
- ✅ **Automated CI/CD pipelines** — push to Git → auto test → auto deploy to production

### Setup:
`User Settings → Git Integration → Connect GitHub / GitLab / Azure DevOps / Bitbucket`

**⚠️ Production Tip:** In any real team setup, always use Git Folders. Set up **branch protection on `main`** so nobody pushes directly to production. Standard workflow:
```
feature branch → PR → code review → merge to dev → tested → merge to main (prod)
```

---

## 12. 🐛 Debugging Databricks Notebooks

### Enable the Interactive Debugger:
`Settings → Developer → Enable Python Notebook Interactive Debugger`
- **Minimum Runtime:** 13.3 LTS or higher
- **Supported Language:** Python only

### Debugging Techniques:

**1. Quick Inspection (Most Common)**
```python
df.show(5)           # see first 5 rows
df.printSchema()     # check column names & data types
df.count()           # row count check
df.describe().show() # quick stats — nulls, min, max, mean
display(df)          # rich visual table in notebook UI
```

**2. Narrow Down with `.limit()` — Save Time**
```python
# Test your transformations on a small slice first, not full data
df.limit(100).display()
```

**3. Interactive Debugger (Breakpoints)**
```python
import pdb
pdb.set_trace()  # execution pauses here — inspect variables interactively
```

**4. Spark UI — For Performance Debugging**
- Go to **Cluster → Spark UI** to see job stages, task durations, shuffle sizes
- Slow stage? Look for **data skew** — one partition much larger than others
- Check for **wide transformations** (joins, groupBy) causing expensive shuffles

**5. Try/Except for Pipeline Stability**
```python
try:
    df = spark.read.parquet("/mnt/data/")
    # your logic
except Exception as e:
    print(f"Pipeline failed: {e}")
    raise  # re-raise so the job actually fails and alerts fire
```

**⚠️ Production Tip:** In production, don't leave `pdb` or excessive `print()` statements. Use the Python `logging` module and write logs to a Delta table or log file. Also always wrap critical steps in `try/except` so failures are caught cleanly and surface proper alerts — not silent crashes.

---

## 📌 Day 2 — Quick Recap

```
Cluster Types     → Single Node (dev/test only) | Multi Node (production)
Access Modes      → Shared = best for production teams (Premium required)
Runtime           → Always use LTS versions in production
Auto Termination  → Aggressive on dev (30-60 min), not needed on job clusters
Auto Scaling      → On-demand driver + Spot workers = cost-efficient & resilient
VM Types          → Match instance type to workload bottleneck, profile first
Cluster Policy    → Enforce in orgs to control costs and governance
Magic Commands    → Cell-level: %sql, %sh, %pip, %run, %fs — interactive use
DBUtils           → Programmatic: secrets, widgets, fs ops, notebook orchestration
Git Folders       → Always in team/prod — branch protection, PRs, CI/CD
Debugging         → .show(), .printSchema(), Spark UI, pdb — use logging in prod
```

---